In [3]:
"""
You can think of this like a "sliding window"
| Shift (s) | Window (T[s:s+4]) |
| ----------| ----------------|
| 0         | T[0:4]          |
| 1         | T[1:5]          |
| 2         | T[2:6]          |
| 3         | T[3:7]          |
| 4         | T[4:8]          |
"""
def naive_string_matcher(T, P):
    n = len(T)
    m = len(P)

    for s in range(n - m + 1):
        if T[s:s + m] == P:
            print("Pattern occurs with shift", s)

In [4]:
T = "AABAACAADAABAABA"
P = "AABA"

naive_string_matcher(T, P)

Pattern occurs with shift 0
Pattern occurs with shift 9
Pattern occurs with shift 12


In [5]:
"""
With P distinct characters:

Every mismatch causes us to skip at least one new character.
If we matched j characters, we skip exactly j positions.
Therefore, each text character is involved in only a constant number of comparisons.
"""
def distinct_naive_matcher(T, P):
    n = len(T)
    m = len(P)

    s = 0

    while s <= n - m:
        j = 0

        while j < m and T[s + j] == P[j]:
            j += 1

        if j == m:
            print("Pattern occurs at shift", s)
            s += m          # all characters are distinct
        elif j == 0:
            s += 1          # first character mismatched
        else:
            s += j          # skip the matched characters

In [6]:
def gap_match(T, P):
    pieces = P.split("<>")
    pos = 0

    for piece in pieces:
        i = T.find(piece, pos)

        if i == -1:
            return False

        pos = i + len(piece)

    return True

In [8]:
"""
Rabin-Karp String Matching

T : text
P : pattern
d : number of possible characters (alphabet size)
q : prime number used for hashing
"""
def rabin_karp_matcher(T, P, d=256, q=101):
    n = len(T)
    m = len(P)

    # d^(m-1) % q
    h = pow(d, m - 1, q)

    p = 0          # hash value of pattern
    t = 0          # hash value of current text window

    # Compute initial hashes
    for i in range(m):
        p = (d * p + ord(P[i])) % q
        t = (d * t + ord(T[i])) % q

    # Slide pattern over text
    for s in range(n - m + 1):
        
        # If hashes match, compare actual strings
        if p == t:
            if P == T[s:s + m]:
                print(f"Pattern occurs at shift {s}")

        # Compute next window hash
        if s < n - m:
            t = (d * (t - ord(T[s]) * h) + ord(T[s + m])) % q

            # Make hash positive
            if t < 0:
                t += q

In [9]:
"""
text = "ABCCDDAEFG"
pattern = "CDD"

Instead of comparing
CDD

against
ABC
BCC
CCD
CDD
DDA
...

character by character every time,
it compares hash values.
Think of a hash as a fingerprint.

"CDD"  ---> hash = 54

"ABC"  ---> hash = 19

"BCC"  ---> hash = 88

"CCD"  ---> hash = 91

"CDD"  ---> hash = 54  <-- Match!

Only when the fingerprints match do we actually compare the strings.


Breaking down the formula
t = (d * (t - ord(T[s]) * h) + ord(T[s + m])) % q

Think of it as four small steps:

Step 1 Remove first character

t - ord(T[s]) * h

Visual ABC

remove A

BC

Step 2 Shift remaining characters

d * (...)

Visual

BC

↓

BC_

Multiplying by d is like shifting the characters one position to the left in a base-d number system.

Step 3 Append new character

+ ord(T[s+m])

Visual

BC_

↓

BCC

Step 4 Keep the number small

% q

Otherwise the hash grows extremely large.

Step 5 — Compare hashes

Suppose

Pattern hash = 82

Window hashes

ABC →59

BCC →74

CCD →31

CDD →82

Hashes match.

Now we verify

if P == T[s:s+m]:
CDD

CDD

Equal!

Output

Pattern occurs at shift 3
"""

'\ntext = "ABCCDDAEFG"\npattern = "CDD"\n\nInstead of comparing\nCDD\n\nagainst\nABC\nBCC\nCCD\nCDD\nDDA\n...\n\ncharacter by character every time,\nit compares hash values.\nThink of a hash as a fingerprint.\n\n"CDD"  ---> hash = 54\n\n"ABC"  ---> hash = 19\n\n"BCC"  ---> hash = 88\n\n"CCD"  ---> hash = 91\n\n"CDD"  ---> hash = 54  <-- Match!\n\nOnly when the fingerprints match do we actually compare the strings.\n'

In [10]:
"""
Think of Rabin-Karp as a fingerprint scanner.

One pattern: Keep one fingerprint and compare every text window against it.
Many patterns of the same length: Keep a set of fingerprints. A window is interesting only if its fingerprint appears in the set.
Many pattern lengths: Keep one fingerprint set per pattern length, and scan the text once for each distinct length.

The rolling hash still lets each window's fingerprint be updated in O(1) time.
The only extra work is maintaining multiple fingerprint sets (or multiple passes when lengths differ),
which is what makes Rabin-Karp extend naturally from one pattern to many.
"""

"\nThink of Rabin-Karp as a fingerprint scanner.\n\nOne pattern: Keep one fingerprint and compare every text window against it.\nMany patterns of the same length: Keep a set of fingerprints. A window is interesting only if its fingerprint appears in the set.\nMany pattern lengths: Keep one fingerprint set per pattern length, and scan the text once for each distinct length.\n\nThe rolling hash still lets each window's fingerprint be updated in O(1) time.\nThe only extra work is maintaining multiple fingerprint sets (or multiple passes when lengths differ),\nwhich is what makes Rabin-Karp extend naturally from one pattern to many.\n"

In [11]:
"""
text: string to search
delta: transition function
pattern_length: len(pattern)
"""
def finite_automaton_matcher(text, delta, pattern_length):
    q = 0
    for i, ch in enumerate(text):
        q = delta(q, ch)

        if q == pattern_length:
            print("Pattern occurs with shift", i - pattern_length + 1)

In [12]:
text = "xxabcxxabc"

pattern = "abc"

m = len(pattern)

delta = {
    0: {'a': 1, 'b': 0, 'c': 0},
    1: {'a': 1, 'b': 2, 'c': 0},
    2: {'a': 1, 'b': 0, 'c': 3},
    3: {'a': 1, 'b': 0, 'c': 0},
}

def transition(state, character):
    return delta[state].get(character, 0)

finite_automaton_matcher(text, transition, m)

Pattern occurs with shift 2
Pattern occurs with shift 7


In [13]:
"""
Computes the transition table δ for the finite automaton matcher.

P: pattern string
alphabet: iterable of characters (e.g. {'a','b'})
"""
def compute_transition_function(P, alphabet):
    m = len(P)
    delta = {}

    for q in range(m + 1):
        delta[q] = {}

        for a in alphabet:
            # Longest possible prefix length
            k = min(m, q + 1)

            # Current matched prefix plus new character
            candidate = P[:q] + a

            # Keep shortening until P[:k] is a suffix
            while k > 0 and not candidate.endswith(P[:k]):
                k -= 1

            delta[q][a] = k

    return delta

In [14]:
pattern = "abab"

alphabet = {'a', 'b'}

delta = compute_transition_function(pattern, alphabet)

for state in delta:
    print(state, delta[state])

0 {'b': 0, 'a': 1}
1 {'b': 2, 'a': 1}
2 {'b': 0, 'a': 3}
3 {'b': 4, 'a': 1}
4 {'b': 0, 'a': 3}


In [15]:
"""
The transition function doesn't ask, "Did the match fail?" It asks,
"After this character, what is the longest prefix of the pattern that I can still claim to have matched?"
The while loop answers that question by repeatedly reducing k until it finds the largest reusable amount of progress.
That's why the automaton can continue scanning the text without ever moving backward.
"""

'\nThe transition function doesn\'t ask, "Did the match fail?" It asks, "After this character,\nwhat is the longest prefix of the pattern that I can still claim to have matched?"\nThe while loop answers that question by repeatedly reducing k until it finds the largest reusable amount of progress.\nThat\'s why the automaton can continue scanning the text without ever moving backward.\n'

In [16]:
def compute_prefix(pattern):
    m = len(pattern)
    pi = [0] * m
    k = 0

    for q in range(1, m):
        while k > 0 and pattern[k] != pattern[q]:
            k = pi[k - 1]

        if pattern[k] == pattern[q]:
            k += 1

        pi[q] = k

    return pi

In [17]:
def kmp_matcher(text, pattern):
    n = len(text)
    m = len(pattern)
    pi = compute_prefix(pattern)
    q = 0

    for i in range(n):
        while q > 0 and pattern[q] != text[i]:
            q = pi[q - 1]

        if pattern[q] == text[i]:
            q += 1

        if q == m:
            print("Pattern found at index", i - m + 1)
            q = pi[q - 1]

In [18]:
"""
T = abcde

The rotations are

abcde
bcdea
cdeab
deabc
eabcd

Now double the string:

abcdeabcde

Every possible rotation appears exactly once as a contiguous substring of length 5:

abcdeabcde
^^^^^

abcdeabcde
 ^^^^^

abcdeabcde
  ^^^^^

abcdeabcde
   ^^^^^

abcdeabcde
    ^^^^^
"""
def is_cyclic_rotation(T, T_prime):

    if len(T) != len(T_prime):
        return False

    S = T + T

    return kmp_matcher(S, T_prime)

In [20]:
"""
Think of the algorithm as repeatedly compressing strings into integers.

Iteration 1: Every 2-character substring gets an integer ID (rank).
Iteration 2: Every 4-character substring becomes a pair of those IDs, which is then assigned a new integer ID.
Iteration 3: Every 8-character substring becomes a pair of 4-character IDs, then gets a new integer ID.
Continue doubling until each suffix has a unique ID.

The left rank identifies the first half of the current substring, and the right rank identifies the second half.
Since those halves were already ranked in the previous iteration, comparing two long substrings reduces to comparing just two integers.
This is what makes the algorithm efficient: instead of repeatedly comparing long strings character by character,
it continually reuses previously computed rankings to build the suffix array in O(n log² n) time
"""
from dataclasses import dataclass

@dataclass
class Suffix:
    index: int
    left_rank: int
    right_rank: int

def make_ranks(sub, rank):
    """
    Assign new ranks after sorting.
    Equal (left_rank, right_rank) pairs receive the same rank.
    """
    current_rank = 1
    rank[sub[0].index] = current_rank

    for i in range(1, len(sub)):
        prev = sub[i - 1]
        curr = sub[i]

        if (curr.left_rank, curr.right_rank) != (prev.left_rank, prev.right_rank):
            current_rank += 1

        rank[curr.index] = current_rank

def compute_suffix_array(T):
    n = len(T)
    suffixes = []
    rank = [0] * n
    SA = [0] * n

    # ------------------------------------
    # Initial ranking using first 2 chars
    # ------------------------------------
    for i in range(n):
        left = ord(T[i])
        right = ord(T[i + 1]) if i + 1 < n else 0
        suffixes.append(Suffix(i, left, right))

    suffixes.sort(key=lambda s: (s.left_rank, s.right_rank))
    length = 2

    while length < n:
        # Give every suffix a rank
        make_ranks(suffixes, rank)

        # Update the rank pairs
        for s in suffixes:
            s.left_rank = rank[s.index]

            if s.index + length < n:
                s.right_rank = rank[s.index + length]
            else:
                s.right_rank = 0

        # Resort according to new ranks
        suffixes.sort(key=lambda s: (s.left_rank, s.right_rank))
        length *= 2

    for i in range(n):
        SA[i] = suffixes[i].index

    return SA

In [21]:
"""
Suffix Array Example: "banana"

Goal:
Sort all suffixes of the string without repeatedly comparing long strings.

------------------------------------------------------------
Original suffixes
------------------------------------------------------------

Index    Suffix
-----    -----------
0        banana
1        anana
2        nana
3        ana
4        na
5        a

Eventually we want:

Index    Suffix
-----    -----------
5        a
3        ana
1        anana
0        banana
4        na
2        nana

Suffix Array = [5, 3, 1, 0, 4, 2]

============================================================
Iteration 1 (length = 2)
============================================================

Look at only the first TWO characters of every suffix.

Index    First 2 chars
-----    -------------
0        ba
1        an
2        na
3        an
4        na
5        a$

($ means end of string and is considered smaller than every letter.)

Represent each substring as a pair of character values:

ba -> (b,a)
an -> (a,n)
na -> (n,a)
an -> (a,n)
na -> (n,a)
a$ -> (a,0)

Sort these pairs:

(a,0)
(a,n)
(a,n)
(b,a)
(n,a)
(n,a)

Assign ranks:

(a,0) -> Rank 1
(a,n) -> Rank 2
(a,n) -> Rank 2
(b,a) -> Rank 3
(n,a) -> Rank 4
(n,a) -> Rank 4

Every suffix now has a rank representing its first 2 characters.

============================================================
Iteration 2 (length = 4)
============================================================

Now compare FOUR characters.

Instead of comparing characters directly, use the ranks from the
previous iteration.

Split each 4-character substring into two 2-character pieces.

Example:

banana

bana
^^^^

becomes

ba | na

We already know:

Rank(ba) = 3
Rank(na) = 4

So "bana" becomes simply

(3,4)

Do this for every suffix.

Example:

Suffix        Representation
------        --------------
banana        (3,4)
anana         (2,2)
nana          (4,1)
ana           (2,0)
na            (4,0)
a             (1,0)

Now sort these integer pairs instead of comparing strings.

Assign NEW ranks after sorting.

These new ranks now represent every 4-character substring.

============================================================
Iteration 3 (length = 8)
============================================================

Now compare EIGHT characters.

Again, split into two ranked halves.

Example:

banana

banana
^^^^^^

Split as

bana | na

The first half already has a rank.
The second half already has a rank.

Instead of comparing

banana

we compare

(rank("bana"), rank("na"))

Again these are just two integers.

Sort.
Assign new ranks.

Now every suffix has a unique rank.

============================================================
Final Result
============================================================

Sorted suffixes:

Rank    Index    Suffix
----    -----    -----------
1       5        a
2       3        ana
3       1        anana
4       0        banana
5       4        na
6       2        nana

Suffix Array:

SA = [5, 3, 1, 0, 4, 2]

============================================================
The Big Idea (The "Aha!" Moment)
============================================================

Iteration 1:
    Rank every 2-character substring.

Iteration 2:
    Build every 4-character substring using two 2-character ranks.

Iteration 3:
    Build every 8-character substring using two 4-character ranks.

Iteration 4:
    Build every 16-character substring using two 8-character ranks.

...

Each iteration DOUBLES how much of the suffix is represented.

Instead of comparing long strings over and over, we compare only

(left_rank, right_rank)

which are just two integers.

That is what makes the suffix array algorithm efficient.
"""

'\nSuffix Array Example: "banana"\n\nGoal:\nSort all suffixes of the string without repeatedly comparing long strings.\n\n------------------------------------------------------------\nOriginal suffixes\n------------------------------------------------------------\n\nIndex    Suffix\n-----    -----------\n0        banana\n1        anana\n2        nana\n3        ana\n4        na\n5        a\n\nEventually we want:\n\nIndex    Suffix\n-----    -----------\n5        a\n3        ana\n1        anana\n0        banana\n4        na\n2        nana\n\nSuffix Array = [5, 3, 1, 0, 4, 2]\n\n============================================================\nIteration 1 (length = 2)\n============================================================\n\nLook at only the first TWO characters of every suffix.\n\nIndex    First 2 chars\n-----    -------------\n0        ba\n1        an\n2        na\n3        an\n4        na\n5        a$\n\n($ means end of string and is considered smaller than every letter.)\n\nReprese

In [22]:
"""
Compute the LCP (Longest Common Prefix) array using Kasai's algorithm.

T  : input string
SA : suffix array

Returns:
    LCP[i] = longest common prefix between
             suffix SA[i] and suffix SA[i-1]
"""
def compute_lcp(T, SA):
    n = len(T)
    rank = [0] * n
    LCP = [0] * n

    # rank[i] tells where suffix i appears in the suffix array
    for i in range(n):
        rank[SA[i]] = i

    l = 0

    for i in range(n):
        # Smallest suffix has no previous neighbor
        if rank[i] == 0:
            continue

        # Previous suffix in sorted order
        j = SA[rank[i] - 1]

        # Count matching characters
        while (
            i + l < n
            and j + l < n
            and T[i + l] == T[j + l]
        ):
            l += 1

        LCP[rank[i]] = l

        # Reuse work for next suffix
        if l > 0:
            l -= 1

    return LCP

In [23]:
"""
Kasai's Algorithm (LCP Array) Example using "banana"

Goal:
Given the suffix array, compute the Longest Common Prefix (LCP) between
every pair of adjacent suffixes in sorted order.

============================================================
Step 1: Build the Suffix Array
============================================================

Original string:

banana

Suffixes:

Index    Suffix
-----    -----------
0        banana
1        anana
2        nana
3        ana
4        na
5        a

Sorted alphabetically:

Position    Original Index    Suffix
--------    --------------    -----------
0           5                 a
1           3                 ana
2           1                 anana
3           0                 banana
4           4                 na
5           2                 nana

Suffix Array:

SA = [5, 3, 1, 0, 4, 2]

============================================================
Step 2: Build the Rank Array
============================================================

The suffix array tells us:

Position 0 -> suffix starting at index 5
Position 1 -> suffix starting at index 3
Position 2 -> suffix starting at index 1
...

Kasai needs the reverse lookup:

"If I'm looking at suffix i,
where does it appear inside the suffix array?"

Build:

rank[SA[i]] = i

Result:

Suffix Index    Rank
------------    ----
0               3
1               2
2               5
3               1
4               4
5               0

Meaning:

Suffix "banana" is at position 3.
Suffix "anana" is at position 2.
Suffix "na" is at position 4.
...

Now we can instantly find a suffix's previous neighbor in sorted order.

============================================================
Step 3: Visit Suffixes in Original Order
============================================================

Notice:

We DO NOT walk through the suffix array.

We walk through the original string:

i = 0
i = 1
i = 2
i = 3
i = 4
i = 5

For each suffix we ask:

"Who comes immediately before me alphabetically?"

============================================================
i = 0
============================================================

Current suffix:

banana

rank[0] = 3

Previous suffix is

SA[2] = 1

which is

anana

Compare:

banana
anana

First letters differ.

Common prefix = ""

Length = 0

LCP[3] = 0

============================================================
i = 1
============================================================

Current suffix:

anana

rank[1] = 2

Previous suffix:

SA[1] = 3

which is

ana

Compare:

anana
ana

Characters:

a = a
n = n
a = a

Stop.

Common prefix:

ana

Length = 3

LCP[2] = 3

============================================================
The Clever Trick
============================================================

Suppose we just discovered

anana
ana

share

ana

Length = 3

Now move one character to the right.

Current suffix becomes

nana

Previous neighbor becomes

na

Notice something:

Removing the first character from both strings gives

nana
na

Since

anana
ana

matched for 3 characters,

we ALREADY KNOW

nana
na

must match for at least

2 characters.

So instead of comparing from the beginning again,

Kasai starts with

l = l - 1

Instead of checking

n
n

a
a

again,

it skips directly to the next unknown character.

============================================================
i = 2
============================================================

Current suffix:

nana

Previous suffix:

na

We already know

na

matches.

Length starts at

2

Only check if there are MORE matching characters.

There aren't.

LCP = 2

============================================================
i = 3
============================================================

Current suffix:

ana

Previous suffix:

a

Common prefix:

a

Length = 1

============================================================
i = 4
============================================================

Current suffix:

na

Previous suffix:

banana

Different first letters.

Length = 0

============================================================
i = 5
============================================================

Current suffix:

a

This is the smallest suffix.

There is no previous suffix.

Skip.

============================================================
Final LCP Array
============================================================

Sorted suffixes:

Position    Suffix
--------    -----------
0           a
1           ana
2           anana
3           banana
4           na
5           nana

Compare adjacent suffixes:

a       vs ana      -> 1
ana     vs anana    -> 3
anana   vs banana   -> 0
banana  vs na       -> 0
na      vs nana     -> 2

Therefore

LCP = [0, 1, 3, 0, 0, 2]

============================================================
Why the Rank Array Matters
============================================================

Suppose we're looking at the suffix

banana

We need to know:

"What suffix comes immediately before banana alphabetically?"

Without rank:

Search through the entire suffix array until you find "banana".

This takes O(n).

With rank:

rank[0] = 3

Immediately know

Previous suffix = SA[2]

This lookup takes O(1).

============================================================
The "Aha!" Moment
============================================================

The suffix array sorts every suffix alphabetically.

The LCP array tells us:

"How many beginning characters do neighboring suffixes share?"

Kasai's brilliant observation is:

If two suffixes share L characters,

then after removing the first character from both,

the next comparison must already share at least L-1 characters.

So instead of restarting every comparison from zero,

Kasai reuses almost all of the previous work by decrementing

l = l - 1

This simple optimization ensures that every character in the string
is compared only a constant number of times, making the entire
algorithm run in O(n) time instead of O(n²).
"""

'\nKasai\'s Algorithm (LCP Array) Example using "banana"\n\nGoal:\nGiven the suffix array, compute the Longest Common Prefix (LCP) between\nevery pair of adjacent suffixes in sorted order.\n\n============================================================\nStep 1: Build the Suffix Array\n============================================================\n\nOriginal string:\n\nbanana\n\nSuffixes:\n\nIndex    Suffix\n-----    -----------\n0        banana\n1        anana\n2        nana\n3        ana\n4        na\n5        a\n\nSorted alphabetically:\n\nPosition    Original Index    Suffix\n--------    --------------    -----------\n0           5                 a\n1           3                 ana\n2           1                 anana\n3           0                 banana\n4           4                 na\n5           2                 nana\n\nSuffix Array:\n\nSA = [5, 3, 1, 0, 4, 2]\n\n============================================================\nStep 2: Build the Rank Array\n=====================

In [24]:
"""
Original String
      │
      ▼
Compute Suffix Array
      │
      ▼
SA = [5,3,1,0,4,2]
      │
      ▼
Build Rank Array
      │
      ▼
rank = [3,2,5,1,4,0]
      │
      ▼
Use SA + Rank to compute LCP
      │
      ▼
LCP = [0,1,3,0,0,2]
------------------------------------------------------------

Think of SA and rank as opposites.

Suffix Array:
Position → Original String Index

SA[2] = 1

"The 3rd smallest suffix starts at index 1."

------------------------------------------------------------

Rank Array:
Original String Index → Position

rank[1] = 2

"The suffix starting at index 1 is the 3rd smallest suffix."

------------------------------------------------------------

Kasai processes suffixes in their ORIGINAL order
(index 0, 1, 2, ...), but it needs to know each suffix's
neighbor in SORTED order.

The rank array is the bridge between those two views.

Without rank:
    Every lookup requires searching the suffix array.
    Time: O(n²)

With rank:
    Every lookup is one array access.
    Time: O(n)
"""

'\nOriginal String\n      │\n      ▼\nCompute Suffix Array\n      │\n      ▼\nSA = [5,3,1,0,4,2]\n      │\n      ▼\nBuild Rank Array\n      │\n      ▼\nrank = [3,2,5,1,4,0]\n      │\n      ▼\nUse SA + Rank to compute LCP\n      │\n      ▼\nLCP = [0,1,3,0,0,2]\n------------------------------------------------------------\n\nThink of SA and rank as opposites.\n\nSuffix Array:\nPosition → Original String Index\n\nSA[2] = 1\n\n"The 3rd smallest suffix starts at index 1."\n\n------------------------------------------------------------\n\nRank Array:\nOriginal String Index → Position\n\nrank[1] = 2\n\n"The suffix starting at index 1 is the 3rd smallest suffix."\n\n------------------------------------------------------------\n\nKasai processes suffixes in their ORIGINAL order\n(index 0, 1, 2, ...), but it needs to know each suffix\'s\nneighbor in SORTED order.\n\nThe rank array is the bridge between those two views.\n\nWithout rank:\n    Every lookup requires searching the suffix array.\n    

In [25]:
"""
We can return early once every suffix has a different rank, the suffix array is completely determined.
More iterations cannot change the order.

Assign new ranks after sorting.
Equal pairs receive the same rank.
Returns the number of distinct ranks.
"""
def make_ranks(sub, rank):
    current_rank = 1
    rank[sub[0].index] = current_rank

    for i in range(1, len(sub)):
        prev = sub[i - 1]
        curr = sub[i]

        if (curr.left_rank, curr.right_rank) != (prev.left_rank, prev.right_rank):
            current_rank += 1

        rank[curr.index] = current_rank

    return current_rank

"""
Modified compute_suffix_array

while length < n:

    num_ranks = make_ranks(suffixes, rank)

    # Stop if every suffix has a unique rank
    if num_ranks == n:
        break

    for s in suffixes:
        s.left_rank = rank[s.index]

        if s.index + length < n:
            s.right_rank = rank[s.index + length]
        else:
            s.right_rank = 0

    suffixes.sort(key=lambda s: (s.left_rank, s.right_rank))

    length *= 2
"""

In [26]:
"""
Best case: ranks separate quickly → stop early.
Worst case: ranks stay tied → must keep doubling.

| Input      | Behavior                                   | Iterations              |
| ---------- | ------------------------------------------ | ----------------------- |
| `abcdefg`  | Every suffix immediately unique            | O(1)                    |
| `aaaaaaaa` | Suffixes remain identical for longest time | Maximum `ceil(log n)-1` |
"""

'\nBest case: ranks separate quickly → stop early.\nWorst case: ranks stay tied → must keep doubling.\n\n| Input      | Behavior                                   | Iterations              |\n| ---------- | ------------------------------------------ | ----------------------- |\n| `abcdefg`  | Every suffix immediately unique            | O(1)                    |\n| `aaaaaaaa` | Suffixes remain identical for longest time | Maximum `ceil(log n)-1` |\n'

In [27]:
"""
A suffix array turns the substring problem into a neighboring-suffix problem.


Normally, to find the longest common substring between two texts,
you'd have to compare every suffix of T₁ with every suffix of T₂, which is quadratic.

The suffix array has already done the hard work of sorting all suffixes.
As a result, suffixes with long common prefixes naturally end up next to each other.
The LCP array then tells you, in constant time per adjacent pair, exactly how long that shared prefix is.

So instead of comparing every pair of suffixes, you simply:
Build one suffix array for T₁#T₂$.
Walk down the suffix array once.
Ignore adjacent suffixes from the same text.
Among adjacent suffixes from different texts, the largest LCP is the longest common substring.

That's why the scan is linear after constructing the suffix array. It isn't that the algorithm somehow "finds"
the common substrings during the scan the suffix array has already organized the suffixes so the answer is sitting
between neighboring entries.


T1 = banana
T2 = ananas

S = T1 + "#" + T2 + "$"

SA = compute_suffix_array(S)

LCP = compute_lcp(S,SA)

best = 0

for i in range(1,len(S)):

    a = SA[i]
    b = SA[i-1]

    if one suffix in T1 and the other in T2:
        best = max(best,LCP[i])

answer = set()

for i in range(1,len(S)):

    a = SA[i]
    b = SA[i-1]

    if one in T1 and one in T2 and LCP[i]==best:
        answer.add(S[a:a+best])

return answer
Why does this work?

Suppose

banana
ananas

Two suffixes are

anana
ananas

They become adjacent in the suffix array because they start almost identically.

Their

LCP = 5

which equals

anana

The algorithm immediately discovers it.
"""

'\nA suffix array turns the substring problem into a neighboring-suffix problem.\n\n\nNormally, to find the longest common substring between two texts,\nyou\'d have to compare every suffix of T₁ with every suffix of T₂, which is quadratic.\n\nThe suffix array has already done the hard work of sorting all suffixes.\nAs a result, suffixes with long common prefixes naturally end up next to each other.\nThe LCP array then tells you, in constant time per adjacent pair, exactly how long that shared prefix is.\n\nSo instead of comparing every pair of suffixes, you simply:\nBuild one suffix array for T₁#T₂$.\nWalk down the suffix array once.\nIgnore adjacent suffixes from the same text.\nAmong adjacent suffixes from different texts, the largest LCP is the longest common substring.\n\nThat\'s why the scan is linear after constructing the suffix array. It isn\'t that the algorithm somehow "finds"\nthe common substrings during the scan the suffix array has already organized the suffixes so the an

In [28]:
"""
The suffix array guarantees:
Every suffix is sorted lexicographically.

It does not guarantee:
Every suffix and its reverse are consecutive.
Only neighboring suffixes are compared in the LCP array.

If another suffix lies between them,

reverse suffix
↓
some unrelated suffix
↓
forward suffix

then the LCP of the forward and reverse suffixes never appears in the LCP array.
The algorithm therefore misses the true longest palindrome.

Intuition
Imagine three suffixes:

abac...
abad...
abac...

The two copies of

Imagine three suffixes:

abac...
abad...
abac...

The two copies of

abac...

are not adjacent because

abad...

falls between them lexicographically.

Since the algorithm only examines adjacent suffixes, it never computes the LCP of the two matching "abac..." suffixes.

What your professor is looking for

Most solutions have two parts:

Provide a counterexample, such as
abacdfgdcaba
Explain the flaw

The algorithm incorrectly assumes that a suffix and the suffix beginning with its reverse will always be adjacent in the suffix array.
This is false because other suffixes with similar prefixes may lie between them. Since the LCP array only stores
longest common prefixes of adjacent suffixes, the true palindrome may never be compared with its reverse,
causing the algorithm to return a palindrome shorter than the actual longest palindrome.
"""

'\nThe suffix array guarantees:\nEvery suffix is sorted lexicographically.\n\nIt does not guarantee:\nEvery suffix and its reverse are consecutive.\nOnly neighboring suffixes are compared in the LCP array.\n\nIf another suffix lies between them,\n\nreverse suffix\n↓\nsome unrelated suffix\n↓\nforward suffix\n\nthen the LCP of the forward and reverse suffixes never appears in the LCP array.\nThe algorithm therefore misses the true longest palindrome.\n\nIntuition\nImagine three suffixes:\n\nabac...\nabad...\nabac...\n\nThe two copies of\n\nImagine three suffixes:\n\nabac...\nabad...\nabac...\n\nThe two copies of\n\nabac...\n\nare not adjacent because\n\nabad...\n\nfalls between them lexicographically.\n\nSince the algorithm only examines adjacent suffixes, it never computes the LCP of the two matching "abac..." suffixes.\n\nWhat your professor is looking for\n\nMost solutions have two parts:\n\nProvide a counterexample, such as\nabacdfgdcaba\nExplain the flaw\n\nThe algorithm incorrectl